assignment build a recommender system and explain how it works

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load data
df = pd.read_csv('products.csv')

# Feature Engineering
df['features'] = df['brand'] + " " + df['category'] + " " + df['cocoa_percent'].astype(str) + "%"

# TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['features'])

# Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Recommendation Function
def recommend_products(product_id, top_n=5):
    idx = df[df['product_id'] == product_id].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recommendations = []
    for i, score in sim_scores:
        recommendations.append({
            'product_id': df.iloc[i]['product_id'],
            'product_name': df.iloc[i]['product_name'],
            'brand': df.iloc[i]['brand'],
            'category': df.iloc[i]['category'],
            'cocoa_percent': df.iloc[i]['cocoa_percent'],
            'similarity_score': round(score, 4)
        })
    
    return pd.DataFrame(recommendations)

# Example Usage
if __name__ == "__main__":
    print("=== Chocolate Product Recommender ===\n")
    sample_id = 'P0001'  # White Chocolate 80%
    print(f"Recommendations for: {df[df['product_id']==sample_id]['product_name'].values[0]}\n")
    
    recs = recommend_products(sample_id, top_n=5)
    print(recs)

=== Chocolate Product Recommender ===

Recommendations for: White Chocolate 80%

  product_id           product_name brand category  cocoa_percent  \
0      P0155     Dark Chocolate 80%  Mars  Truffle             80   
1      P0012  Truffle Chocolate 80%  Mars  Praline             80   
2      P0067  Praline Chocolate 80%  Mars  Praline             80   
3      P0105     Milk Chocolate 80%  Mars    White             80   
4      P0113  Truffle Chocolate 80%  Mars    White             80   

   similarity_score  
0            1.0000  
1            0.6899  
2            0.6899  
3            0.6821  
4            0.6821  


In [8]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
df = pd.read_csv('products.csv')
print(f"Loaded {len(df)} products")
print(df.head())

Loaded 200 products
  product_id           product_name    brand category  cocoa_percent  weight_g
0      P0001    White Chocolate 80%     Mars  Truffle             80       120
1      P0002     Dark Chocolate 70%  Cadbury  Praline             70       100
2      P0003  Truffle Chocolate 70%  Hershey  Praline             70       120
3      P0004     Milk Chocolate 50%     Mars  Praline             50        80
4      P0005    White Chocolate 70%  Ferrero    White             70        50


In [13]:
# ========================= LOAD DATA =========================
df = pd.read_csv('products.csv')
print(f"Loaded {len(df)} products")

# ========================= BUILD RECOMMENDER =========================
np.random.seed(42)
num_users = 100
user_ids = [f"U{str(i).zfill(3)}" for i in range(1, num_users + 1)]

ratings = []
for user in user_ids:
    num_ratings = np.random.randint(20, 50)
    sampled_products = df.sample(n=num_ratings, random_state=np.random.randint(1000))
    for _, prod in sampled_products.iterrows():
        rating = np.random.randint(1, 6)
        ratings.append({
            'user_id': user,
            'product_id': prod['product_id'],
            'rating': rating
        })

ratings_df = pd.DataFrame(ratings)
print(f"Generated {len(ratings_df)} ratings from {num_users} users")

# Create user-item matrix
user_item_matrix = ratings_df.pivot_table(
    index='user_id', 
    columns='product_id', 
    values='rating'
).fillna(0)

# Item-item similarity
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity, 
    index=user_item_matrix.columns, 
    columns=user_item_matrix.columns
)

# ========================= RECOMMENDATION FUNCTION =========================
def get_recommendations(user_id, n_recommendations=5):
    if user_id not in user_item_matrix.index:
        return "User not found"
    
    user_ratings = user_item_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index
    
    scores = pd.Series(0.0, index=user_item_matrix.columns)
    for item in rated_items:
        sim_scores = item_similarity_df.loc[item]
        scores += sim_scores * user_ratings[item]
    
    scores = scores.drop(rated_items, errors='ignore')
    top_recs = scores.nlargest(n_recommendations)
    
    recommendations = df[df['product_id'].isin(top_recs.index)].copy()
    recommendations = recommendations.set_index('product_id')
    recommendations['predicted_rating'] = top_recs.round(2)
    
    return recommendations[['product_name', 'brand', 'category', 'cocoa_percent', 'predicted_rating']]

# ========================= TEST =========================
print("\n" + "="*60)
print("SAMPLE RECOMMENDATIONS FOR USER U001")
print("="*60)
print(get_recommendations('U001', 5))

Loaded 200 products
Generated 3481 ratings from 100 users

SAMPLE RECOMMENDATIONS FOR USER U001
                     product_name    brand category  cocoa_percent  \
product_id                                                           
P0029          Milk Chocolate 60%    Lindt     Milk             60   
P0037       Truffle Chocolate 60%  Hershey    White             60   
P0124       Truffle Chocolate 80%    Lindt  Praline             80   
P0183         White Chocolate 50%     Mars  Praline             50   
P0195         White Chocolate 90%  Cadbury     Dark             90   

            predicted_rating  
product_id                    
P0029                  16.92  
P0037                  17.77  
P0124                  17.00  
P0183                  18.34  
P0195                  18.64  


In [14]:
!pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 598.5 kB/s eta 0:00:02
   ----------------------- ---------------- 0.8/1.3 MB 657.8 kB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 825.2 kB/s eta 0:00:01
   ---------------------------------------  1.3/1.3 MB 871.6 kB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 862.6 kB/s eta 0:00:00


In [15]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# ========================= LOAD PRODUCTS =========================
# (Same as before - creates products.csv if needed)
csv_data = """product_id,product_name,brand,category,cocoa_percent,weight_g
P0001,White Chocolate 80%,Mars,Truffle,80,120
... [same full CSV data as in previous message] ..."""  # (I'll keep it short here - use the full one from before)

with open('products.csv', 'w', encoding='utf-8') as f:
    f.write(csv_data)

products = pd.read_csv('products.csv')
print(f"Loaded {len(products)} products")

# ========================= GENERATE SYNTHETIC RATINGS =========================
np.random.seed(42)
num_users = 100
user_ids = [f"U{str(i).zfill(3)}" for i in range(1, num_users + 1)]

ratings = []
for user in user_ids:
    num_ratings = np.random.randint(20, 50)
    sampled = products.sample(n=num_ratings, random_state=np.random.randint(1000))
    for _, prod in sampled.iterrows():
        rating = np.random.randint(1, 6)
        ratings.append({
            'user_id': user,
            'product_id': prod['product_id'],
            'rating': rating
        })

ratings_df = pd.DataFrame(ratings)
print(f"Generated {len(ratings_df)} ratings")

# ========================= MATRIX FACTORIZATION WITH SURPRISE =========================
# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'product_id', 'rating']], reader)

# Split into train/test
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD (Matrix Factorization)
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

# Evaluate
predictions = svd.test(testset)
print("RMSE:", accuracy.rmse(predictions))

# ========================= RECOMMENDATION FUNCTION =========================
def get_mf_recommendations(user_id, n=5):
    """Get top N recommendations using Matrix Factorization"""
    if user_id not in ratings_df['user_id'].unique():
        return "User not found"
    
    # Get all products
    all_products = products['product_id'].unique()
    
    # Products already rated by user
    rated = ratings_df[ratings_df['user_id'] == user_id]['product_id'].unique()
    
    # Predict ratings for unrated products
    predictions = []
    for pid in all_products:
        if pid not in rated:
            pred = svd.predict(user_id, pid)
            predictions.append((pid, pred.est))
    
    # Sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions[:n]
    
    # Get product details
    top_ids = [x[0] for x in top_n]
    recs = products[products['product_id'].isin(top_ids)].copy()
    recs = recs.set_index('product_id')
    
    # Add predicted scores
    pred_dict = dict(top_n)
    recs['predicted_rating'] = [pred_dict[pid] for pid in recs.index]
    
    return recs[['product_name', 'brand', 'category', 'cocoa_percent', 'predicted_rating']].round(2)

# ========================= TEST =========================
print("\n" + "="*70)
print("MATRIX FACTORIZATION RECOMMENDATIONS FOR USER U001")
print("="*70)
print(get_mf_recommendations('U001', 5))

Loaded 2 products


ValueError: Cannot take a larger sample than population when 'replace=False'